In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, matthews_corrcoef

In [2]:
df = pd.read_csv("morganFP_1024.csv")
df

,0,1,2,3,4,5,6,7,8,9,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
756,0,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
757,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
758,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
df_class = pd.read_csv("ido_tdo.csv" , index_col=0)
df_class

,ido_tdo
0,AA
1,II
2,AI
3,II
4,II
...,...
755,II
756,II
757,IA
758,AI


In [4]:
y = df_class.iloc[:,-1]
y

0      AA
1      II
2      AI
3      II
4      II
       ..
755    II
756    II
757    IA
758    AI
759    AA
Name: ido_tdo, Length: 760, dtype: object

In [5]:
X = df.iloc[:, :]
X

,0,1,2,3,4,5,6,7,8,9,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
756,0,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
757,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
758,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
#Random split:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42 , test_size=0.2)

In [8]:
model = SVC()
params = {
    "C": [0.1, 1, 10, 100, 1000],
    "gamma": ["scale", "auto", 1, 0.1, 0.01, 0.001],
    "kernel": ["linear", "rbf", "poly", "sigmoid"]
}
grid = GridSearchCV(model, params, cv=5, scoring='matthews_corrcoef', n_jobs=-1)
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100, 1000],
                         'gamma': ['scale', 'auto', 1, 0.1, 0.01, 0.001],
                         'kernel': ['linear', 'rbf', 'poly', 'sigmoid']},
             scoring='matthews_corrcoef')

In [8]:
#For random-splitting
grid.best_estimator_

SVC(C=10, gamma=0.01, kernel='sigmoid')

In [10]:
clf = SVC(probability=True, C=10, gamma=0.01, kernel='sigmoid', random_state=42)

In [11]:
clf.fit(X_train, y_train)

SVC(C=10, gamma=0.01, kernel='sigmoid', probability=True, random_state=42)

In [12]:
ypred = clf.predict(X_test)

In [13]:
print("svm morgan 1024")
print(classification_report(y_test , ypred))
print(confusion_matrix(y_test , ypred))
print(roc_auc_score(y_test, clf.predict_proba(X_test) , multi_class='ovr'))
print(matthews_corrcoef(y_test, ypred))

svm morgan 1024
              precision    recall  f1-score   support

          AA       0.62      0.69      0.65        42
          AI       0.76      0.74      0.75        34
          IA       0.50      0.38      0.43        16
          II       0.80      0.80      0.80        60

    accuracy                           0.71       152
   macro avg       0.67      0.65      0.66       152
weighted avg       0.71      0.71      0.71       152

[[29  8  2  3]
 [ 4 25  1  4]
 [ 5  0  6  5]
 [ 9  0  3 48]]
0.8876280519179099
0.5883495719686417


In [14]:
#External test:
df_test = pd.read_csv("test_morganFP_1024.csv")
df_test_class = pd.read_csv("testdata_target.csv")
y_t = df_test_class.iloc[:,-1]
X_t = df_test.iloc[:, :]
yt_pred = clf.predict(X_t)
print("svm")
print(classification_report(y_t , yt_pred))
print(confusion_matrix(y_t , yt_pred))
print(roc_auc_score(y_t, clf.predict_proba(X_t) , multi_class='ovr'))
print(matthews_corrcoef(y_t, yt_pred))

svm
              precision    recall  f1-score   support

          AA       0.00      0.00      0.00         2
          AI       0.60      0.22      0.32        27
          IA       0.12      0.12      0.12         8
          II       0.69      0.86      0.77        74

    accuracy                           0.64       111
   macro avg       0.35      0.30      0.30       111
weighted avg       0.61      0.64      0.60       111

[[ 0  0  1  1]
 [ 0  6  0 21]
 [ 0  0  1  7]
 [ 0  4  6 64]]
0.6740818252832657
0.14437299358926375


/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

# **Using class_weight='balanced'**

In [15]:
model = SVC(class_weight='balanced')
params = {
    "C": [0.1, 1, 10, 100, 1000],
    "gamma": ["scale", "auto", 1, 0.1, 0.01, 0.001],
    "kernel": ["linear", "rbf", "poly", "sigmoid"]
}
grid = GridSearchCV(model, params, cv=5, scoring='matthews_corrcoef', n_jobs=-1)
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SVC(class_weight='balanced'), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100, 1000],
                         'gamma': ['scale', 'auto', 1, 0.1, 0.01, 0.001],
                         'kernel': ['linear', 'rbf', 'poly', 'sigmoid']},
             scoring='matthews_corrcoef')

In [16]:
grid.best_estimator_

SVC(C=0.1, class_weight='balanced', kernel='linear')

In [21]:
clf = SVC(probability=True, C=0.1, class_weight='balanced', kernel='linear', random_state=42)

In [22]:
clf.fit(X_train, y_train)

SVC(C=0.1, class_weight='balanced', kernel='linear', probability=True,
    random_state=42)

In [23]:
ypred = clf.predict(X_test)

In [24]:
print("svm morgan 1024, balanced")
print(classification_report(y_test , ypred))
print(confusion_matrix(y_test , ypred))
print(roc_auc_score(y_test, clf.predict_proba(X_test) , multi_class='ovr'))
print(matthews_corrcoef(y_test, ypred))

svm morgan 1024, balanced
              precision    recall  f1-score   support

          AA       0.63      0.69      0.66        42
          AI       0.76      0.74      0.75        34
          IA       0.42      0.50      0.46        16
          II       0.85      0.77      0.81        60

    accuracy                           0.71       152
   macro avg       0.67      0.67      0.67       152
weighted avg       0.72      0.71      0.72       152

[[29  8  3  2]
 [ 5 25  0  4]
 [ 6  0  8  2]
 [ 6  0  8 46]]
0.8948784493029476
0.5959569555935964


In [25]:
#External test:
df_test = pd.read_csv("test_morganFP_1024.csv")
df_test_class = pd.read_csv("testdata_target.csv")
y_t = df_test_class.iloc[:,-1]
X_t = df_test.iloc[:, :]
yt_pred = clf.predict(X_t)
print("svm")
print(classification_report(y_t , yt_pred))
print(confusion_matrix(y_t , yt_pred))
print(roc_auc_score(y_t, clf.predict_proba(X_t) , multi_class='ovr'))
print(matthews_corrcoef(y_t, yt_pred))

svm
              precision    recall  f1-score   support

          AA       0.00      0.00      0.00         2
          AI       0.67      0.15      0.24        27
          IA       0.00      0.00      0.00         8
          II       0.69      0.96      0.80        74

    accuracy                           0.68       111
   macro avg       0.34      0.28      0.26       111
weighted avg       0.62      0.68      0.59       111

[[ 0  0  1  1]
 [ 0  4  0 23]
 [ 0  0  0  8]
 [ 0  2  1 71]]
0.6999695984924394
0.16509551510219184


/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

# **SMOTE - Synthetic Minority Over-sampling Technique**

In [26]:
from imblearn.over_sampling import SMOTE

In [27]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [28]:
model = SVC()
params = {
    "C": [0.1, 1, 10, 100, 1000],
    "gamma": ["scale", "auto", 1, 0.1, 0.01, 0.001],
    "kernel": ["linear", "rbf", "poly", "sigmoid"]
}
grid = GridSearchCV(model, params, cv=5, scoring='matthews_corrcoef', n_jobs=-1)
grid.fit(X_train_smote, y_train_smote)

GridSearchCV(cv=5, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100, 1000],
                         'gamma': ['scale', 'auto', 1, 0.1, 0.01, 0.001],
                         'kernel': ['linear', 'rbf', 'poly', 'sigmoid']},
             scoring='matthews_corrcoef')

In [29]:
grid.best_estimator_

SVC(C=100)

In [30]:
clf = SVC(probability=True, C=100, random_state=42)

In [31]:
clf.fit(X_train_smote, y_train_smote)

SVC(C=100, probability=True, random_state=42)

In [32]:
ypred = clf.predict(X_test)

In [33]:
print("svm morgan 2048")
print(classification_report(y_test , ypred))
print(confusion_matrix(y_test , ypred))
print(roc_auc_score(y_test, clf.predict_proba(X_test) , multi_class='ovr'))
print(matthews_corrcoef(y_test, ypred))

svm morgan 2048
              precision    recall  f1-score   support

          AA       0.64      0.69      0.67        42
          AI       0.72      0.68      0.70        34
          IA       0.54      0.44      0.48        16
          II       0.82      0.85      0.84        60

    accuracy                           0.72       152
   macro avg       0.68      0.66      0.67       152
weighted avg       0.72      0.72      0.72       152

[[29  9  2  2]
 [ 5 23  0  6]
 [ 6  0  7  3]
 [ 5  0  4 51]]
0.8751081232909355
0.6063875859642105


In [34]:
#External test:
df_test = pd.read_csv("test_morganFP_1024.csv")
df_test_class = pd.read_csv("testdata_target.csv")
y_t = df_test_class.iloc[:,-1]
X_t = df_test.iloc[:, :]
yt_pred = clf.predict(X_t)
print("svm")
print(classification_report(y_t , yt_pred))
print(confusion_matrix(y_t , yt_pred))
print(roc_auc_score(y_t, clf.predict_proba(X_t) , multi_class='ovr'))
print(matthews_corrcoef(y_t, yt_pred))

svm
              precision    recall  f1-score   support

          AA       0.00      0.00      0.00         2
          AI       0.75      0.22      0.34        27
          IA       0.00      0.00      0.00         8
          II       0.70      0.97      0.81        74

    accuracy                           0.70       111
   macro avg       0.36      0.30      0.29       111
weighted avg       0.65      0.70      0.63       111

[[ 0  0  0  2]
 [ 0  6  0 21]
 [ 0  0  0  8]
 [ 0  2  0 72]]
0.6952089213364167
0.25973433107898725


/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

# **Oversample using Adaptive Synthetic (ADASYN) algorithm**

In [29]:
from imblearn.over_sampling import ADASYN

In [30]:
ada = ADASYN(random_state=42)
X_train_ada, y_train_ada = smote.fit_resample(X_train, y_train)

In [31]:
model = SVC()
params = {
    "C": [0.1, 1, 10, 100, 1000],
    "gamma": ["scale", "auto", 1, 0.1, 0.01, 0.001],
    "kernel": ["linear", "rbf", "poly", "sigmoid"]
}
grid = GridSearchCV(model, params, cv=5, n_jobs=-1)
grid.fit(X_train_ada, y_train_ada)

GridSearchCV(cv=5, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100, 1000],
                         'gamma': ['scale', 'auto', 1, 0.1, 0.01, 0.001],
                         'kernel': ['linear', 'rbf', 'poly', 'sigmoid']})

In [32]:
grid.best_estimator_

SVC(C=100)

In [33]:
clf = SVC(probability=True, C=100, random_state=42)

In [34]:
clf.fit(X_train_ada, y_train_ada)

SVC(C=100, probability=True, random_state=42)

In [35]:
ypred = clf.predict(X_test)

In [36]:
print("svm morgan 1024")
print(classification_report(y_test , ypred))
print(confusion_matrix(y_test , ypred))
print(roc_auc_score(y_test, clf.predict_proba(X_test) , multi_class='ovr'))
print(matthews_corrcoef(y_test, ypred))

svm morgan 2048
              precision    recall  f1-score   support

          AA       0.64      0.69      0.67        42
          AI       0.72      0.68      0.70        34
          IA       0.54      0.44      0.48        16
          II       0.82      0.85      0.84        60

    accuracy                           0.72       152
   macro avg       0.68      0.66      0.67       152
weighted avg       0.72      0.72      0.72       152

[[29  9  2  2]
 [ 5 23  0  6]
 [ 6  0  7  3]
 [ 5  0  4 51]]
0.8751081232909355
0.6063875859642105


In [37]:
#External test:
df_test = pd.read_csv("test_morganFP_1024.csv")
df_test_class = pd.read_csv("testdata_target.csv")
y_t = df_test_class.iloc[:,-1]
X_t = df_test.iloc[:, :]
yt_pred = clf.predict(X_t)
print("svm")
print(classification_report(y_t , yt_pred))
print(confusion_matrix(y_t , yt_pred))
print(roc_auc_score(y_t, clf.predict_proba(X_t) , multi_class='ovr'))
print(matthews_corrcoef(y_t, yt_pred))

svm
              precision    recall  f1-score   support

          AA       0.00      0.00      0.00         2
          AI       0.75      0.22      0.34        27
          IA       0.00      0.00      0.00         8
          II       0.70      0.97      0.81        74

    accuracy                           0.70       111
   macro avg       0.36      0.30      0.29       111
weighted avg       0.65      0.70      0.63       111

[[ 0  0  0  2]
 [ 0  6  0 21]
 [ 0  0  0  8]
 [ 0  2  0 72]]
0.6952089213364167
0.25973433107898725


/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

# **For Bemis-Murcko scaffold-splitting, run the following cells:**

In [6]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

In [7]:
df_total = pd.read_csv('final_total.csv')
smiles_list = [df_total.loc[i, 'smiles'] for i in range(len(df_total))]

In [8]:
def generate_scaffold(smiles, include_chirality=False):
    """
    Generate the Bemis-Murcko scaffold for a given SMILES string.
    
    Parameters:
    -----------
    smiles : str
        SMILES representation of the molecule
    include_chirality : bool
        Whether to include chirality in the scaffold
    
    Returns:
    --------
    str : Scaffold SMILES string
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return smiles
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(
            mol=mol, 
            includeChirality=include_chirality
        )
        return scaffold
    except:
        return smiles

def scaffold_split(X, y, smiles_list, frac_train=0.8, frac_test=0.2, seed=None):
    """
    Split data by Bemis-Murcko scaffolds.
    
    Parameters:
    -----------
    X : array-like
        Feature matrix (fingerprints)
    y : array-like
        Target labels
    smiles_list : list
        List of SMILES strings for each molecule
    frac_train : float
        Fraction of data for training
    frac_test : float
        Fraction of data for test
    seed : int or None
        Random seed for reproducibility
    
    Returns:
    --------
    tuple : (X_train, X_test, y_train, y_test, 
             idx_train, idx_test)
    """
    n_samples = len(smiles_list)
    
    # Group molecules by scaffold
    scaffolds = defaultdict(list)
    for idx, smiles in enumerate(smiles_list):
        scaffold = generate_scaffold(smiles, include_chirality=True)
        scaffolds[scaffold].append(idx)
    
    # Sort scaffolds by size (largest first)
    sorted_scaffolds = sorted(
        scaffolds.items(), 
        key=lambda x: len(x[1]), 
        reverse=True
    )
    
    # Calculate target sizes
    train_cutoff = int(frac_train * n_samples)
    
    train_idx = []
    test_idx = []
    
    # Assign entire scaffold groups to splits
    for scaffold, indices in sorted_scaffolds:
        if len(train_idx) + len(indices) <= train_cutoff:
            train_idx.extend(indices)
        else:
            test_idx.extend(indices)
    
    # Convert to numpy arrays if needed
    X = np.array(X)
    y = np.array(y)
    
    # Create splits
    X_train = X[train_idx]
    X_test = X[test_idx]
    
    y_train = y[train_idx]
    y_test = y[test_idx]
    
    print(f"Scaffold split results:")
    print(f"  Training:   {len(train_idx)} molecules ({len(train_idx)/n_samples*100:.1f}%)")
    print(f"  Test:       {len(test_idx)} molecules ({len(test_idx)/n_samples*100:.1f}%)")
    print(f"  Unique scaffolds: {len(scaffolds)}")
    
    return (X_train, X_test, 
            y_train, y_test,
            train_idx, test_idx)

In [9]:
# Perform scaffold split
X_train, X_test, y_train, y_test, idx_train, idx_test = scaffold_split(
    X, 
    y, 
    smiles_list,
    frac_train=0.8,
    frac_test=0.2,
    seed=42
)

print(f"\nShape of splits:")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_test: {X_test.shape}, y_test: {y_test.shape}")

Scaffold split results:
  Training:   608 molecules (80.0%)
  Test:       152 molecules (20.0%)
  Unique scaffolds: 250

Shape of splits:
  X_train: (608, 1024), y_train: (608,)
  X_test: (152, 1024), y_test: (152,)


In [10]:
model = SVC()
params = {
    "C": [0.1, 1, 10, 100, 1000],
    "gamma": ["scale", "auto", 1, 0.1, 0.01, 0.001],
    "kernel": ["linear", "rbf", "poly", "sigmoid"]
}
grid = GridSearchCV(model, params, cv=5, scoring='matthews_corrcoef', n_jobs=-1)
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100, 1000],
                         'gamma': ['scale', 'auto', 1, 0.1, 0.01, 0.001],
                         'kernel': ['linear', 'rbf', 'poly', 'sigmoid']},
             scoring='matthews_corrcoef')

In [11]:
grid.best_estimator_

SVC(C=1)

In [12]:
clf = SVC(probability=True, C=1, random_state=42)
clf.fit(X_train, y_train)
ypred = clf.predict(X_test)

In [13]:
print("svm morgan 1024")
print(classification_report(y_test , ypred))
print(confusion_matrix(y_test , ypred))
print(roc_auc_score(y_test, clf.predict_proba(X_test) , multi_class='ovr'))
print(matthews_corrcoef(y_test, ypred))

svm morgan 1024
              precision    recall  f1-score   support

          AA       0.56      0.71      0.62        35
          AI       0.83      0.77      0.80        39
          IA       0.25      0.09      0.13        11
          II       0.72      0.72      0.72        67

    accuracy                           0.68       152
   macro avg       0.59      0.57      0.57       152
weighted avg       0.68      0.68      0.67       152

[[25  1  0  9]
 [ 6 30  0  3]
 [ 3  0  1  7]
 [11  5  3 48]]
0.8431385326962598
0.5348386543777172


In [14]:
#External test:
df_test = pd.read_csv("test_morganFP_1024.csv")
df_test_class = pd.read_csv("testdata_target.csv")
y_t = df_test_class.iloc[:,-1]
X_t = df_test.iloc[:, :]
yt_pred = clf.predict(X_t)
print("svm")
print(classification_report(y_t , yt_pred))
print(confusion_matrix(y_t , yt_pred))
print(roc_auc_score(y_t, clf.predict_proba(X_t) , multi_class='ovr'))
print(matthews_corrcoef(y_t, yt_pred))

svm
              precision    recall  f1-score   support

          AA       0.00      0.00      0.00         2
          AI       0.00      0.00      0.00        27
          IA       0.00      0.00      0.00         8
          II       0.67      1.00      0.80        74

    accuracy                           0.67       111
   macro avg       0.17      0.25      0.20       111
weighted avg       0.44      0.67      0.53       111

[[ 0  0  0  2]
 [ 0  0  0 27]
 [ 0  0  0  8]
 [ 0  0  0 74]]
0.6597429306961539
0.0
